# Прогон — Mask R-CNN **v2** (`feature/mask-rcnn` @ `07e109c+`)

Ветка Даши: Mask R-CNN v2 (ResNet-50-FPN + GroupNorm, elongated anchors, `enforce_disjoint`).
Артефакты: `/kaggle/working/outputs/mask_rcnn/`.

**Settings**
- Accelerator: **GPU T4** (или P100)
- Internet: **on** (pip)
- Add Data: competition `filament-segmentation-2026` **и** dataset `dm1triyaleksandrov/mask-rcnn-v2-src` (repo.zip; private GitHub clone не нужен)
- Fallback: Secret `GITHUB_TOKEN` если dataset не прикреплён

Дефолты конфига: `architecture=v2`, `image_size=1280`, `epochs=30`, `lr=2e-4`, `enforce_disjoint=True`.
Старые v1 веса (`best.pt` @1024) **несовместимы** — полный retrain.

`python run.py` = CLI Mask2Former. Этот ноутбук зовёт `src.models.mask_rcnn.train` напрямую.

Сначала `SMOKE = True` (1 эпоха). Потом `SMOKE = False` и Save Version **с Output**.

In [ ]:
import os, sys, subprocess, zipfile, shutil
from pathlib import Path

REPO = Path("/kaggle/working/filament-segmentation-2026")

def _find_src_root() -> Path | None:
    """Dataset was uploaded with --dir-mode zip, so files live unpacked under /kaggle/input/..."""
    print("input tree:", sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else None)
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]:
        if not root.exists():
            continue
        for run_py in root.rglob("run.py"):
            parent = run_py.parent
            if (parent / "src/models/mask_rcnn/train.py").exists():
                candidates.append(parent)
        for zpath in root.rglob("repo.zip"):
            candidates.append(zpath)
    print("src candidates:", candidates)
    return candidates[0] if candidates else None

def _materialize(src: Path):
    if REPO.exists():
        shutil.rmtree(REPO)
    REPO.mkdir(parents=True, exist_ok=True)
    if src.suffix == ".zip":
        with zipfile.ZipFile(src) as zf:
            zf.extractall(REPO)
        if not (REPO / "run.py").exists():
            nested = [p for p in REPO.iterdir() if p.is_dir() and (p / "run.py").exists()]
            if nested:
                for item in nested[0].iterdir():
                    shutil.move(str(item), str(REPO / item.name))
                shutil.rmtree(nested[0])
    else:
        # copytree contents
        for item in src.iterdir():
            dest = REPO / item.name
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)
    print("materialized", src, "->", REPO)

def _clone_git():
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        sec = UserSecretsClient()
        for name in ("GITHUB_TOKEN", "GH_TOKEN", "GIT_TOKEN"):
            try:
                token = sec.get_secret(name) or ""
            except Exception:
                token = ""
            if token:
                break
    except Exception as e:
        print("secrets unavailable:", type(e).__name__, e)
    if not token:
        raise SystemExit(
            "No mask-rcnn source under /kaggle/input and no GITHUB_TOKEN. "
            "Attach dataset dm1triyaleksandrov/mask-rcnn-v2-src."
        )
    branch = os.environ.get("GIT_BRANCH", "feature/mask-rcnn")
    url = "https://github.com/d1maal/filament-segmentation-2026.git".replace(
        "https://", f"https://x-access-token:{token}@", 1
    )
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", branch, url, str(REPO)])
    print("cloned", branch)

src = _find_src_root()
if src is not None:
    _materialize(src)
elif not (REPO / "run.py").exists():
    _clone_git()
else:
    print("reuse existing", REPO)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("cwd", Path.cwd())
assert (REPO / "src/models/mask_rcnn/train.py").exists(), "mask_rcnn package missing"
print("architecture file ok:", (REPO / "src/models/mask_rcnn/config.py").read_text().split("architecture")[1][:40])

!pip -q install python-dotenv opencv-python-headless pycocotools tqdm

**GPU**

In [ ]:
import torch
import torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Settings → GPU T4, Restart session"
x = torch.zeros(1, device="cuda")
print("ok", x.device)

**TRAIN (v2)**

Дефолты Даши v2: `architecture=v2`, `epochs=30`, `image_size=1280`, `lr=2e-4`, AMP + OneCycle + photometric aug.
На T4 полный val (~387) каждый эпох может не влезть в 12ч — при `SMOKE=False` держим `EVAL_LIMIT=50`.
После train — отдельный full-val в ячейке ниже (defaults postprocess: score=0.5, min_area=150, `enforce_disjoint=True`).
Не передавай `resume=` от v1 `best.pt`.

In [ ]:
from pathlib import Path
from src.models.mask_rcnn.config import config as mrcnn_cfg
from src.models.mask_rcnn.train import train

SMOKE = False  # True → 1 epoch smoke; False → full v2 train
EPOCHS = 1 if SMOKE else int(mrcnn_cfg.epochs)          # v2 default 30
IMAGE_SIZE = int(mrcnn_cfg.image_size)                  # v2 default 1280
EVAL_LIMIT = 8 if SMOKE else 50                         # full val after train

mrcnn_cfg.output_dir = Path("/kaggle/working/outputs/mask_rcnn")
mrcnn_cfg.output_dir.mkdir(parents=True, exist_ok=True)
print("out:", mrcnn_cfg.output_dir)
print(
    f"arch={mrcnn_cfg.architecture} smoke={SMOKE} epochs={EPOCHS} "
    f"size={IMAGE_SIZE} lr={mrcnn_cfg.lr} amp={mrcnn_cfg.amp} "
    f"enforce_disjoint={mrcnn_cfg.enforce_disjoint} eval_limit={EVAL_LIMIT}"
)
assert mrcnn_cfg.architecture == "v2", "Expected Mask R-CNN v2 config"

!python run.py check
best = train(epochs=EPOCHS, image_size=IMAGE_SIZE, eval_limit=EVAL_LIMIT)
print("best:", best)
!ls -lh /kaggle/working/outputs/mask_rcnn/
!head -5 /kaggle/working/outputs/mask_rcnn/train_log.csv || true

**METRICS**

In [ ]:
from pathlib import Path

RUN = Path("/kaggle/working/outputs/mask_rcnn")
print((RUN / "train_log.csv").read_text() if (RUN / "train_log.csv").exists() else "no train_log.csv")
print("last", (RUN / "last.pt").exists(), "best", (RUN / "best.pt").exists())
print("epoch ckpts:", sorted((RUN / "weights").glob("*.pt")) if (RUN / "weights").exists() else [])

**VAL PQ + SUBMISSION (v2 defaults)**

Использует `predict_masks` с конфигом v2: score=0.5, min_area=150, `enforce_disjoint=True` (disjoint внутри infer — отдельный noolap не нужен).
Перед ячейкой Save Version **с Output**, если сессия может оборваться.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm import tqdm

from src.config import config
from src.data import instances_for_image, mask_to_rle_counts, three_channel
from src.eval.submission import _write_csv
from src.eval.val_items import _val_items
from src.metrics import aggregate, panoptic_quality
from src.models.mask_rcnn.config import config as mrcnn_cfg
from src.models.mask_rcnn.infer import load_checkpoint, predict_masks

WEIGHTS = Path("/kaggle/working/outputs/mask_rcnn/best.pt")
# None → use v2 config defaults (score=0.5, min_area=150, enforce_disjoint=True)
SCORE_THR = None
MIN_AREA = None
ENFORCE_DISJOINT = True
SUBMIT = True

def to_tensor(gray: np.ndarray) -> torch.Tensor:
    rgb = three_channel(gray)
    return torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0

def _overlap_count(masks):
    if len(masks) < 2:
        return 0
    stack = np.stack([(m > 0) for m in masks]).astype(np.uint8)
    return int((stack.sum(axis=0) > 1).sum())

model, device = load_checkpoint(WEIGHTS)
items = _val_items()
rows = []
overlap_pixels = 0
print(
    f"infer val: {len(items)} | score={SCORE_THR or mrcnn_cfg.score_threshold} "
    f"| min_area={MIN_AREA or mrcnn_cfg.min_area} | disjoint={ENFORCE_DISJOINT}",
    flush=True,
)
for coco, info, path in tqdm(items, desc="val"):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise SystemExit(f"Failed to read image: {path}")
    pred = predict_masks(
        model, to_tensor(gray), device,
        score_threshold=SCORE_THR, min_area=MIN_AREA,
        enforce_disjoint=ENFORCE_DISJOINT,
    )
    overlap_pixels += _overlap_count(pred)
    metrics = panoptic_quality(instances_for_image(coco, info["id"]), pred)
    metrics["stem"] = Path(info["file_name"]).stem
    rows.append(metrics)

summary = aggregate(rows)
print("Home PQ (Mask R-CNN v2):", round(summary["pq"], 4), summary)
print("Overlap pixels across val (must be 0):", overlap_pixels)
assert overlap_pixels == 0, "enforce_disjoint failed — do not submit overlapping masks"
out = Path("/kaggle/working/outputs/mask_rcnn")
out.mkdir(parents=True, exist_ok=True)
(out / "val_metrics.json").write_text(
    json.dumps(
        {
            "summary": summary,
            "per_image": rows,
            "weights": str(WEIGHTS),
            "architecture": mrcnn_cfg.architecture,
            "enforce_disjoint": ENFORCE_DISJOINT,
            "overlap_pixels": overlap_pixels,
        },
        indent=2,
    ),
    encoding="utf-8",
)

if SUBMIT:
    csv_rows = []
    test_overlap = 0
    test_images = sorted(
        p for p in config.test_images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    print(f"test: {len(test_images)} images", flush=True)
    for path in tqdm(test_images, desc="test"):
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise SystemExit(f"Failed to read image: {path}")
        pred = predict_masks(
            model, to_tensor(gray), device,
            score_threshold=SCORE_THR, min_area=MIN_AREA,
            enforce_disjoint=ENFORCE_DISJOINT,
        )
        test_overlap += _overlap_count(pred)
        for i, mask in enumerate(pred, start=1):
            csv_rows.append(
                {"filament_id": f"{path.stem}_{i}", "segmentation_rle": mask_to_rle_counts(mask)}
            )
    print("Overlap pixels across test (must be 0):", test_overlap)
    assert test_overlap == 0, "test masks overlap — abort submit"
    _write_csv(out / "submission_mask_rcnn_v2.csv", csv_rows)
    !ls -lh /kaggle/working/outputs/mask_rcnn/submission_mask_rcnn_v2.csv